# Filipino Tokenizer — Usage Guide

`filipino-tokenizer` is a morphology-aware BPE tokenizer for Philippine languages.  
It combines a structured Filipino affix database with Byte-Pair Encoding, ensuring  
that BPE merges **never cross morpheme boundaries**.

## Installation

```bash
pip install filipino-tokenizer           # core library
pip install filipino-tokenizer[hf]       # + HuggingFace Transformers integration
```

## What's in this notebook

| # | Section | What you'll learn |
|---|---------|-------------------|
| 1 | Train | Train a tokenizer on a Filipino corpus |
| 2 | Encode / Decode | Convert text ↔ token IDs |
| 3 | Tokenize | Inspect subword tokens and morpheme boundaries |
| 4 | Morphological Segmenter | Decompose words into morphemes |
| 5 | Save & Reload | Persist and share a trained tokenizer |
| 6 | Real corpus | Load a tokenizer trained on Wikitext-TL-39 |
| 7 | HuggingFace | Use with `transformers` |
| 8 | SLM / LLM Training | Feed into a language model |

In [ ]:
import os
import tempfile

from filipino_tokenizer.tagalog import TagalogTokenizer, TagalogSegmenter

## 1. Train the Tokenizer

`TagalogTokenizer.train()` reads a plain-text corpus (one sentence per line),  
morphologically segments every word, then trains constrained BPE on top.

For this demo we use a small built-in corpus so the cell runs in seconds.  
See **Section 6** for loading a tokenizer trained on the full Wikitext-TL-39 dataset.

In [ ]:
CORPUS = """\
Kumain siya ng pagkain sa hapagkainan.
Ang mga bata ay masayang naglalaro sa labas ng bahay.
Maganda ang panahon ngayon kaya lumabas kami para maglakad.
Bumili ako ng mga prutas sa palengke kahapon.
Nagluluto ang nanay ng masarap na adobo para sa buong pamilya.
Pumunta kami sa simbahan tuwing Linggo ng umaga.
Nagbabasa ang mga estudyante ng libro sa silid-aklatan.
Kumanta ang mga bata sa programa ng paaralan nila.
Umuulan kaya nagdala ako ng payong at kapote.
Naglinis kami ng bahay bago dumating ang mga bisita.
Nagtatrabaho ang tatay sa opisina araw-araw.
Kinain niya ang lahat ng pagkain sa mesa.
Pinakamahusay ang ginawa niya sa pagtatanghal.
Natutulog na ang sanggol sa kuna niya.
Gumagawa siya ng takdang-aralin bago matulog.
"""

tmpdir = tempfile.mkdtemp()
corpus_path = os.path.join(tmpdir, "corpus.txt")
with open(corpus_path, "w", encoding="utf-8") as f:
    f.write(CORPUS)

tok = TagalogTokenizer()
tok.train(corpus_path, vocab_size=500)

print(f"Vocabulary : {len(tok.bpe.vocab):,} tokens")
print(f"Merges     : {len(tok.bpe.merges):,}")

## 2. Encode & Decode

`encode(text)` → `list[int]` — token IDs, the exact format expected by neural network models.  
`decode(ids)` → `str` — reconstructs the original text (lowercased).

In [ ]:
text = "Kumain siya ng pagkain."

ids     = tok.encode(text)
decoded = tok.decode(ids)

print(f"Input   : {text}")
print(f"IDs     : {ids}")
print(f"Decoded : {decoded}")
print(f"Match   : {decoded == text.lower()}")

## 3. Tokenize — Inspect Subword Tokens

`tokenize(text)` returns the string representation of each BPE token.  
The `▁` marker shows **morpheme boundaries** — BPE is never allowed to merge across them.

For example, `kumain` (to eat) has infix `-um-` inserted after the first consonant:  
surface form → `k▁um▁ain` (three morpheme-safe segments).

In [ ]:
examples = [
    "kumain",                                       # infix -um-  → k▁um▁ain
    "pagkain",                                      # prefix pag- → pag▁kain
    "pinakamahusay",                                # stacked     → pinaka▁ma▁husay
    "Nagluluto ang nanay ng masarap na adobo.",     # full sentence
]

for text in examples:
    tokens = tok.tokenize(text)
    print(f"  {text}")
    print(f"  → {tokens}")
    print()

## 4. Morphological Segmenter

`TagalogSegmenter` can be used standalone to decompose Filipino words into morphemes.  
This is the linguistic step that runs *before* BPE — it tells the tokenizer where boundaries are.

Handles:
- **Prefixes** — `pag-`, `mag-`, `pinaka-`, stacked prefixes
- **Infixes** — `-um-`, `-in-`
- **Suffixes** — `-an`, `-in`, `-han`
- **Circumfixes** — `ka-…-han`, `pag-…-an`
- **Frozen forms & loan words** — returned unsegmented

In [ ]:
seg = TagalogSegmenter()

words = [
    ("kumain",        "infix -um-"),
    ("pagkain",       "prefix pag-"),
    ("kinain",        "infix -in-"),
    ("pagkainan",     "circumfix pag-…-an"),
    ("pinakamahusay", "stacked prefixes"),
    ("nagtatrabaho",  "prefix nag- + loan root"),
    ("kasiyahan",     "circumfix ka-…-han"),
    ("pangalan",      "frozen form"),
    ("computer",      "loan word"),
]

print(f"  {'Word':<20} {'Type':<24} Morphemes")
print(f"  {'-'*20} {'-'*24} ---------")
for word, kind in words:
    morphemes = seg.segment(word)
    print(f"  {word:<20} {kind:<24} {morphemes}")

## 5. Save & Reload

A trained tokenizer is saved as two plain-text files:
- `vocab.json` — token string → integer ID mapping  
- `merges.txt` — ordered BPE merge rules, one per line

These files are portable — share them so others can load your tokenizer without retraining.

In [ ]:
save_dir = os.path.join(tmpdir, "my_tokenizer")
tok.save(save_dir)

print(f"Saved to: {save_dir}/")
for fname in ["vocab.json", "merges.txt"]:
    size = os.path.getsize(os.path.join(save_dir, fname))
    print(f"  {fname:<12} {size:,} bytes")

# Reload into a fresh instance
tok2 = TagalogTokenizer()
tok2.load(save_dir)

sample = "Nagtatrabaho ang tatay sa opisina."
assert tok.encode(sample) == tok2.encode(sample)
print("\nReloaded tokenizer produces identical output: OK")

## 6. Load a Tokenizer Trained on the Real Corpus

The repository includes a pretrained tokenizer trained on [Wikitext-TL-39](https://huggingface.co/datasets/linkanjarad/Wikitext-TL39) (~1.5M sentences):
- **32,000 token vocabulary**
- **31,900 merge rules**
- Trained in ~2.6 minutes on a modern CPU

To train your own from scratch:
```bash
python scripts/download_corpus.py                           # one-time download
python examples/training_tagalog_tokenizer.py               # ~3 min, saves to demo/models/morph/
```

In [ ]:
# Notebook lives in demo/ so models/morph is a sibling directory
MODEL_DIR = os.path.join(os.path.dirname(os.getcwd()), "demo", "models", "morph")
# If running from project root: MODEL_DIR = "demo/models/morph"

if os.path.isdir(MODEL_DIR):
    tok_full = TagalogTokenizer()
    tok_full.load(MODEL_DIR)
    print(f"Loaded pretrained tokenizer from {MODEL_DIR}")
    print(f"  Vocabulary : {len(tok_full.bpe.vocab):,} tokens")
    print(f"  Merges     : {len(tok_full.bpe.merges):,}")

    # Compare output with the demo tokenizer
    sample = "Nagluluto ang nanay ng masarap na adobo."
    print(f"\nSample: {sample!r}")
    print(f"  Demo tokens  ({len(tok.bpe.vocab):,} vocab)  : {tok.tokenize(sample)}")
    print(f"  Full tokens  ({len(tok_full.bpe.vocab):,} vocab) : {tok_full.tokenize(sample)}")
else:
    print(f"Pretrained model not found at {MODEL_DIR}")
    print("Run: python scripts/download_corpus.py && python examples/training_tagalog_tokenizer.py")
    tok_full = tok  # fall back to demo tokenizer for remaining sections

## 7. HuggingFace Integration

`TagalogHFTokenizer` wraps the tokenizer behind the `PreTrainedTokenizer` interface,  
making it compatible with the full HuggingFace ecosystem:  
**Trainer**, **TRL**, **Axolotl**, **LlamaFactory**, and any other `transformers`-based pipeline.

```bash
pip install filipino-tokenizer[hf]
```

In [ ]:
try:
    from filipino_tokenizer.tagalog import TagalogHFTokenizer

    hf_tok = TagalogHFTokenizer(
        vocab_file=os.path.join(MODEL_DIR, "vocab.json"),
        merges_file=os.path.join(MODEL_DIR, "merges.txt"),
    )
    print(f"Vocab size : {hf_tok.vocab_size:,}")
    print(f"BOS token  : {hf_tok.bos_token!r}  (id {hf_tok.bos_token_id})")
    print(f"EOS token  : {hf_tok.eos_token!r}  (id {hf_tok.eos_token_id})")
    print(f"PAD token  : {hf_tok.pad_token!r}  (id {hf_tok.pad_token_id})")

    # Standard HuggingFace call — returns input_ids + attention_mask
    sentences = [
        "Kumain siya ng pagkain.",
        "Nagtatrabaho ang tatay sa opisina araw-araw.",
    ]
    encoding = hf_tok(sentences, padding=True, truncation=True, return_tensors="pt")
    print(f"\nBatch encoding (padded to same length):")
    print(f"  input_ids shape      : {list(encoding['input_ids'].shape)}")
    print(f"  attention_mask shape : {list(encoding['attention_mask'].shape)}")

    # Save in HuggingFace format (adds tokenizer_config.json + special_tokens_map.json)
    hf_save_dir = os.path.join(tmpdir, "hf_tokenizer")
    hf_tok.save_pretrained(hf_save_dir)
    print(f"\nSaved HF tokenizer to: {hf_save_dir}/")
    print("  Files:", sorted(os.listdir(hf_save_dir)))

    # Reload with from_pretrained
    hf_tok2 = TagalogHFTokenizer.from_pretrained(hf_save_dir)
    assert hf_tok.encode("Kumain") == hf_tok2.encode("Kumain")
    print("  from_pretrained reload: OK")

except ImportError:
    print("transformers not installed.")
    print("Run: pip install filipino-tokenizer[hf]")

## 8. Use in SLM / LLM Training

Once you have a `TagalogHFTokenizer`, it plugs into **any HuggingFace-compatible training framework**.  
The tokenizer is model-agnostic — swap any architecture you want:

| Architecture | Use case |
|---|---|
| `GPT2LMHeadModel` | Text generation, causal LM (simplest to start) |
| `LlamaForCausalLM` | Modern SLM / LLM training |
| `BertForMaskedLM` | Classification, NER, masked LM |
| `T5ForConditionalGeneration` | Translation, summarization |

The only model-specific setting from the tokenizer is **`vocab_size`**.

In [ ]:
try:
    import torch
    from torch.utils.data import Dataset
    from transformers import GPT2Config, GPT2LMHeadModel

    # --- 1. Tokenize a dataset ---
    class FilipinoTextDataset(Dataset):
        def __init__(self, texts, tokenizer, max_length=128):
            self.encodings = tokenizer(
                texts,
                max_length=max_length,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )

        def __len__(self):
            return self.encodings["input_ids"].shape[0]

        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.encodings.items()}
            item["labels"] = item["input_ids"].clone()   # causal LM: predict next token
            return item

    texts = [
        "Kumain siya ng pagkain sa hapagkainan.",
        "Ang mga bata ay masayang naglalaro sa labas ng bahay.",
        "Nagtatrabaho ang tatay sa opisina araw-araw.",
        "Maganda ang panahon ngayon kaya lumabas kami para maglakad.",
    ]
    dataset = FilipinoTextDataset(texts, hf_tok)
    sample = dataset[0]
    print(f"Dataset samples    : {len(dataset)}")
    print(f"input_ids shape    : {list(sample['input_ids'].shape)}")
    print(f"labels shape       : {list(sample['labels'].shape)}")

    # --- 2. Build a model with the right vocab_size ---
    config = GPT2Config(
        vocab_size=hf_tok.vocab_size,   # must match your tokenizer
        n_embd=256,                     # embedding dim  (use 768+ for real training)
        n_layer=4,                      # transformer layers (use 12+ for real training)
        n_head=4,                       # attention heads
        pad_token_id=hf_tok.pad_token_id,
        bos_token_id=hf_tok.bos_token_id,
        eos_token_id=hf_tok.eos_token_id,
    )
    model = GPT2LMHeadModel(config)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel              : GPT-2 (small config for demo)")
    print(f"Parameters         : {total_params / 1e6:.1f}M")
    print(f"Embedding matrix   : {hf_tok.vocab_size:,} × {config.n_embd} = {hf_tok.vocab_size * config.n_embd:,} params")

    # --- 3. One forward pass to verify shapes ---
    batch = dataset[:]  # all samples
    with torch.no_grad():
        out = model(**{k: v for k, v in batch.items() if k != "labels"},
                    labels=batch["labels"])
    print(f"\nForward pass loss  : {out.loss.item():.4f}  (random init, expected ~log(vocab_size) = {torch.log(torch.tensor(hf_tok.vocab_size)):.2f})")

    # --- 4. Swap in any other architecture ---
    print("\n--- Swap to a different architecture ---")
    print("from transformers import LlamaConfig, LlamaForCausalLM")
    print(f"config = LlamaConfig(vocab_size={hf_tok.vocab_size}, ...)")
    print("model  = LlamaForCausalLM(config)")
    print("# Everything else (Dataset, Trainer, etc.) stays the same.")

except (ImportError, NameError):
    print("Requires: pip install filipino-tokenizer[hf] torch")
    print("Section 7 must also run successfully first.")

## Next Steps

| Resource | Description |
|----------|-------------|
| `tokenizer_comparisons_fil.ipynb` | Side-by-side comparison vs GPT-2 BPE and SentencePiece on Filipino sentences |
| `tokenizer_comparisons.ipynb` | Full evaluation on the Wikitext-TL-39 corpus with charts |
| `scripts/download_corpus.py` | Download Wikitext-TL-39 for real training |
| `examples/training_tagalog_tokenizer.py` | Production training script with live progress reporting |
| [GitHub Issues](https://github.com/JpCurada/filipino-tokenizer/issues) | Report bugs or request features |